# NLP + Finance: FinBERT

so i've been using a general-purpose TF-IDF approach this whole time. while researching i found a model called FinBERT — it was actually trained specifically on financial text, unlike my bag-of-words approach. wanted to see if domain-specific training actually makes a difference.

the efficient market hypothesis says sentiment shouldn't predict returns cleanly. but if it does contain any signal, a model that actually understands financial language should extract it better than one that just counts words. that's the question here.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# loading finbert — pretrained on financial text unlike general purpose models
nlp = pipeline("text-classification", model="ProsusAI/finbert")
print("FinBERT loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded.


In [3]:
# load the dataset and preprocess — same as before
import pandas as pd
from src.preprocess import preprocess

df = pd.read_csv('../data/raw/all-data.csv', encoding='latin-1', header=None)
df.columns = ['sentiment', 'headline']
df['cleaned'] = df['headline'].apply(preprocess)

print(df.shape)
print(df['sentiment'].value_counts())

(4846, 3)
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


In [4]:
# run finbert on a sample — full dataset takes too long on CPU
sample = df.sample(500, random_state=42).reset_index(drop=True)

# handle 512 token truncation
def get_finbert_sentiment(text):
    result = nlp(text[:512])[0]
    return result['label'].lower(), result['score']

sample[['finbert_label', 'finbert_score']] = sample['headline'].apply(
    lambda x: pd.Series(get_finbert_sentiment(x))
)

print(sample['finbert_label'].value_counts())
print(sample.head(3))

finbert_label
neutral     268
positive    162
negative     70
Name: count, dtype: int64
  sentiment                                           headline  \
0   neutral  The company was supposed to deliver machinery ...   
1   neutral  UNC Charlotte would also deploy SSH Tectia Con...   
2   neutral  In 2009 , Lee & Man had a combined annual prod...   

                                             cleaned finbert_label  \
0  company supposed deliver machinery veneer mill...       neutral   
1  unc charlotte would also deploy ssh tectia con...       neutral   
2  lee man combined annual production capacity cl...       neutral   

   finbert_score  
0       0.931113  
1       0.580098  
2       0.920752  


In [5]:
from sklearn.metrics import f1_score, classification_report
import joblib

# load saved vectorizer and linearsvc
vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')
lsvc = joblib.load('../models/linearsvc_model.pkl')

# get linearsvc predictions on same sample
X_sample = vectorizer.transform(sample['cleaned'])
sample['lsvc_label'] = lsvc.predict(X_sample)

# compare weighted F1
finbert_f1 = f1_score(sample['sentiment'], sample['finbert_label'], average='weighted')
lsvc_f1 = f1_score(sample['sentiment'], sample['lsvc_label'], average='weighted')

print(f"FinBERT weighted F1: {finbert_f1:.4f}")
print(f"LinearSVC weighted F1: {lsvc_f1:.4f}")

FinBERT weighted F1: 0.9009
LinearSVC weighted F1: 0.9238


hm. LinearSVC is beating FinBERT here : 0.92 vs 0.90. i wasn't expecting that. a few things could explain this: the sample is small (500 headlines), FinBERT was trained on a different distribution of financial text, and the phrasebank dataset is already pretty clean and domain-specific so TF-IDF might be enough. this doesn't mean FinBERT is worse in general - just on this particular dataset and sample.

In [6]:
# per-class F1 comparison
print("FinBERT per-class F1:")
print(classification_report(sample['sentiment'], sample['finbert_label']))

print("LinearSVC per-class F1:")
print(classification_report(sample['sentiment'], sample['lsvc_label']))

FinBERT per-class F1:
              precision    recall  f1-score   support

    negative       0.83      1.00      0.91        58
     neutral       0.96      0.87      0.92       295
    positive       0.83      0.91      0.87       147

    accuracy                           0.90       500
   macro avg       0.87      0.93      0.90       500
weighted avg       0.91      0.90      0.90       500

LinearSVC per-class F1:
              precision    recall  f1-score   support

    negative       0.91      0.84      0.88        58
     neutral       0.94      0.95      0.94       295
    positive       0.91      0.91      0.91       147

    accuracy                           0.92       500
   macro avg       0.92      0.90      0.91       500
weighted avg       0.92      0.92      0.92       500



looking at the per-class numbers - FinBERT actually does better on negative (F1 0.91 vs 0.88) which makes sense, it understands context and negation better. but LinearSVC dominates on neutral and positive. the negative class is the most important one for a trading system - missing a negative headline is the dangerous failure mode. so FinBERT has an edge where it actually matters most.

In [7]:
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

sample['vader_label'] = sample['headline'].apply(get_vader_sentiment)
vader_f1 = f1_score(sample['sentiment'], sample['vader_label'], average='weighted')
print(f"VADER weighted F1: {vader_f1:.4f}")

VADER weighted F1: 0.5499


In [8]:
# three-way comparison table
comparison = pd.DataFrame({
    'Model': ['VADER', 'LinearSVC', 'FinBERT'],
    'Weighted F1': [vader_f1, lsvc_f1, finbert_f1],
    'Negative F1': [
        f1_score(sample['sentiment'], sample['vader_label'], average=None, labels=['negative', 'neutral', 'positive'])[0],
        f1_score(sample['sentiment'], sample['lsvc_label'], average=None, labels=['negative', 'neutral', 'positive'])[0],
        f1_score(sample['sentiment'], sample['finbert_label'], average=None, labels=['negative', 'neutral', 'positive'])[0]
    ],
    'Neutral F1': [
        f1_score(sample['sentiment'], sample['vader_label'], average=None, labels=['negative', 'neutral', 'positive'])[1],
        f1_score(sample['sentiment'], sample['lsvc_label'], average=None, labels=['negative', 'neutral', 'positive'])[1],
        f1_score(sample['sentiment'], sample['finbert_label'], average=None, labels=['negative', 'neutral', 'positive'])[1]
    ],
    'Positive F1': [
        f1_score(sample['sentiment'], sample['vader_label'], average=None, labels=['negative', 'neutral', 'positive'])[2],
        f1_score(sample['sentiment'], sample['lsvc_label'], average=None, labels=['negative', 'neutral', 'positive'])[2],
        f1_score(sample['sentiment'], sample['finbert_label'], average=None, labels=['negative', 'neutral', 'positive'])[2]
    ]
})

print(comparison.round(4).to_string(index=False))

    Model  Weighted F1  Negative F1  Neutral F1  Positive F1
    VADER       0.5499       0.2857      0.6166       0.5202
LinearSVC       0.9238       0.8750      0.9410       0.9085
  FinBERT       0.9009       0.9062      0.9165       0.8673


In [9]:
from scipy.stats import spearmanr

# load AAPL data and compute IC for FinBERT signals
# reusing the merged dataframe approach from 03_trading.ipynb
aapl = pd.read_csv('../data/raw/all-data.csv', encoding='latin-1', header=None)

In [10]:
# find examples where finbert is correct and linearsvc is wrong
finbert_wins = sample[
    (sample['finbert_label'] == sample['sentiment']) & 
    (sample['lsvc_label'] != sample['sentiment'])
]

print(f"cases where FinBERT correct, LinearSVC wrong: {len(finbert_wins)}")
print(finbert_wins[['headline', 'sentiment', 'finbert_label', 'lsvc_label']].head(10))

cases where FinBERT correct, LinearSVC wrong: 30
                                              headline sentiment  \
6    Finnish-owned contract manufacturer of electro...  positive   
20   According to the Latvian business register , U...  negative   
36   Nokia s U.S. shares were 3.3 percent lower at ...  negative   
47   In 2006 , TeliaSonera net sales were SEK 91 bn...   neutral   
60   This new deal has strengthened the partnership...  positive   
80   The Internal Revenue Service sees about 20 per...   neutral   
110  The plant will go on stream in November 2008 a...   neutral   
114  TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sel...  positive   
123  treatment products in Usa , Canada , Mexico , ...   neutral   
148  Raute posted a net profit of 1.8 mln euro $ 2....  positive   

    finbert_label lsvc_label  
6        positive    neutral  
20       negative    neutral  
36       negative    neutral  
47        neutral   positive  
60       positive    neutral  
80        neutra

## Where FinBERT Gets It Right

30 cases where FinBERT got it right and LinearSVC didn't. looked through them and picked the most interesting ones to understand why.

In [12]:
# printing full headlines for the interesting cases
interesting_idx = [6, 20, 36, 110]
for idx in interesting_idx:
    row = finbert_wins[finbert_wins.index == idx].iloc[0]
    print(f"Headline: {row['headline']}")
    print(f"True label: {row['sentiment']}")
    print(f"FinBERT: {row['finbert_label']} | LinearSVC: {row['lsvc_label']}")
    print()

Headline: Finnish-owned contract manufacturer of electronics Elcoteq Hungary Kft has announced plans to recruit more than 650 new staffers to fulfill new orders in P+_cs , where the company has two plants .
True label: positive
FinBERT: positive | LinearSVC: neutral

Headline: According to the Latvian business register , Uponor Latvia closed in red with LVL 99,000 EUR 139,538.17 USD 194,556.48 on turnover of LVL 2.346 mn for 2009 .
True label: negative
FinBERT: negative | LinearSVC: neutral

Headline: Nokia s U.S. shares were 3.3 percent lower at $ 12.73 by 1750 GMT .
True label: negative
FinBERT: negative | LinearSVC: neutral

Headline: The plant will go on stream in November 2008 and its estimated daily production will be 120,000 litres of bioethanol .
True label: neutral
FinBERT: neutral | LinearSVC: negative



went through the cases and a few things stood out.

"closed in red with LVL 99,000" gets called neutral by LinearSVC. makes sense actually, it sees numbers and country names but has no idea "closed in red" means a loss. FinBERT was trained on this kind of financial language so it gets it.

"Nokia shares were 3.3 percent lower" is neutral again from LinearSVC. "lower" probably shows up in all kinds of contexts in training. but 3.3 percent lower on a stock is clearly bad news and FinBERT picks up on that.

"plans to recruit more than 650 new staffers to fulfill new orders" gets missed as positive by LinearSVC. hiring to fulfill new orders is a good signal for a company but "recruit" and "staffers" aren't inherently positive words. FinBERT gets the business context.

"plant will go on stream in November 2008" gets called negative by LinearSVC, probably some word got flagged. it's just neutral operational news and FinBERT gets it right.

so, the bag of words struggles with financial idioms and context. not surprising but good to see it concretely.